# EU Pesticides — Extract Names + SMILES

**Strategy:**
1. Parse MRL XML files (`pesticides/Publication1-8.xml`) → ~679 unique substance names
2. Load Active Substance Excel (`ActiveSubstanceExport_03-04-2026.xlsx`) → 1480 entries with CAS numbers
3. Match names → get CAS numbers
4. Query PubChem via `pubchempy` by CAS → get SMILES

In [1]:
import xml.etree.ElementTree as ET
import glob
import re
import pandas as pd
import pubchempy as pcp

## Step 1 — Extract substance names from MRL XML files

In [2]:
xml_files = sorted(glob.glob('pesticides/Publication*.xml'))

xml_names = set()
for path in xml_files:
    tree = ET.parse(path)
    root = tree.getroot()
    for s in root.findall('Substances'):
        name_el = s.find('Name')
        if name_el is not None and name_el.text:
            xml_names.add(name_el.text.strip())

print(f'Unique substance names across all XML files: {len(xml_names)}')
sorted(xml_names)[:10]

Unique substance names across all XML files: 679


['1,1-dichloro-2,2-bis(4-ethylphenyl)ethane  (F)',
 '1,2-dibromoethane (ethylene dibromide) (F)',
 '1,2-dichloroethane (ethylene dichloride) (F)',
 '1,3-Dichloropropene',
 '1,4-Diaminobutane  (aka Putrescine) (++)',
 '1,4-dimethylnaphthalene (R)(F)',
 '1-Decanol',
 '1-Naphthylacetamide and 1-naphthylacetic acid (sum of 1-naphthylacetamide and 1-naphthylacetic acid and its salts, expressed as 1-naphythlacetic acid)',
 '1-methyl-3-(trifluoromethyl)-1H-pyrazole-4-carboxamide (PAM)',
 '1-methylcyclopropene']

## Step 2 — Load Active Substance Excel

In [3]:
df_excel = pd.read_excel('ActiveSubstanceExport_03-04-2026.xlsx', header=2)
df_excel = df_excel[['Substance', 'CAS Number']].dropna(subset=['Substance'])
df_excel.columns = ['substance', 'cas']
df_excel['substance'] = df_excel['substance'].str.strip()
df_excel['cas'] = df_excel['cas'].astype(str).str.strip()

print(f'Excel rows: {len(df_excel)}')
df_excel.head()

Excel rows: 1478


,substance,cas
0,(3E)-dec-3-en-2-one,10519-33-2
1,"(4Z-9Z)-7,9-Dodecadien-1-ol",No CAS allocated
2,(E)-10-Dodecen-1-yl acetate,No CAS allocated
3,(E)-11-Tetradecen-1-yl acetate (SCLP Acetates),33189-72-9
4,"(E)-2-Methyl-6-methylene-2,7-octadien-1-ol (my...",543-39-5


## Step 3 — Match XML names to Excel (get CAS numbers)

XML names often carry MRL qualifiers like `(F)`, `(R)`, `(++)`, or descriptions in parentheses. We strip them before matching.

In [4]:
def strip_qualifiers(name):
    """Repeatedly strip trailing parenthetical qualifiers."""
    cleaned = name
    while True:
        new = re.sub(r'\s*\([^()]*\)\s*$', '', cleaned).strip()
        if new == cleaned:
            break
        cleaned = new
    return cleaned

# Build lookup: lowercased substance name -> (original name, CAS)
excel_lookup = {row['substance'].lower(): (row['substance'], row['cas'])
                for _, row in df_excel.iterrows()}

all_substances = []  # will include both matched and unmatched

for xml_name in sorted(xml_names):
    # 1. Exact match
    if xml_name.lower() in excel_lookup:
        orig, cas = excel_lookup[xml_name.lower()]
        all_substances.append({'xml_name': xml_name, 'name': orig, 'cas': cas, 'matched': True})
        continue
    # 2. After stripping qualifiers
    cleaned = strip_qualifiers(xml_name)
    if cleaned.lower() in excel_lookup:
        orig, cas = excel_lookup[cleaned.lower()]
        all_substances.append({'xml_name': xml_name, 'name': orig, 'cas': cas, 'matched': True})
        continue
    # 3. No match — keep anyway, PubChem will be tried by name
    all_substances.append({'xml_name': xml_name, 'name': cleaned or xml_name, 'cas': None, 'matched': False})

df_all = pd.DataFrame(all_substances)
print(f'Total substances:   {len(df_all)}')
print(f'Matched to Excel:   {df_all["matched"].sum()}')
print(f'No Excel match:     {(~df_all["matched"]).sum()} (PubChem name lookup will be tried)')
df_all.head(10)


Total substances:   679
Matched to Excel:   493
No Excel match:     186 (PubChem name lookup will be tried)


,xml_name,name,cas,matched
0,"1,1-dichloro-2,2-bis(4-ethylphenyl)ethane (F)","1,1-dichloro-2,2-bis(4-ethylphenyl)ethane",NaN,False
1,"1,2-dibromoethane (ethylene dibromide) (F)","1,2-Dibromoethane",106-93-4,True
2,"1,2-dichloroethane (ethylene dichloride) (F)","1,2-Dichloroethane",107-06-2,True
3,"1,3-Dichloropropene","1,3-Dichloropropene",542-75-6,True
4,"1,4-Diaminobutane (aka Putrescine) (++)","1,4-Diaminobutane",NaN,False
5,"1,4-dimethylnaphthalene (R)(F)","1,4-Dimethylnaphthalene",571-58-4,True
6,1-Decanol,1-Decanol,112-30-1,True
7,1-Naphthylacetamide and 1-naphthylacetic acid ...,1-Naphthylacetamide and 1-naphthylacetic acid,NaN,False
8,1-methyl-3-(trifluoromethyl)-1H-pyrazole-4-car...,1-methyl-3-(trifluoromethyl)-1H-pyrazole-4-car...,NaN,False
9,1-methylcyclopropene,1-methylcyclopropene,3100-04-7,True


In [5]:
# Substances not matched in Excel — inspect before PubChem lookup
df_all[~df_all['matched']][['xml_name', 'name']]


,xml_name,name
0,"1,1-dichloro-2,2-bis(4-ethylphenyl)ethane (F)","1,1-dichloro-2,2-bis(4-ethylphenyl)ethane"
4,"1,4-Diaminobutane (aka Putrescine) (++)","1,4-Diaminobutane"
7,1-Naphthylacetamide and 1-naphthylacetic acid ...,1-Naphthylacetamide and 1-naphthylacetic acid
8,1-methyl-3-(trifluoromethyl)-1H-pyrazole-4-car...,1-methyl-3-(trifluoromethyl)-1H-pyrazole-4-car...
14,"2-amino-4-methoxy-6-(trifluormethyl)-1,3,5-tri...","2-amino-4-methoxy-6-(trifluormethyl)-1,3,5-tri..."
...,...,...
668,Verticillium albo-atrum isolate WCS850,Verticillium albo-atrum isolate WCS850
671,Warfarin,Warfarin
673,"Z,Z,Z,Z-7,13,16,19-docosatetraen-1-yl isobutyr...","Z,Z,Z,Z-7,13,16,19-docosatetraen-1-yl isobutyrate"
674,Z-13-hexadecen-11-yn-1-yl acetate (F),Z-13-hexadecen-11-yn-1-yl acetate


## Step 4 — Query PubChem for SMILES

Uses `pubchempy` which wraps the PubChem PUG REST API.  
- Tries lookup by CAS number first (most reliable)
- Falls back to substance name for unmatched entries  
- Biological agents (Bacillus, Trichoderma...) and complex mixtures will return `None` — expected


In [ ]:
def fetch_smiles(cas, name):
    """Return IsomericSMILES from PubChem. Tries CAS first, then name."""
    cas_valid = bool(re.match(r'^\d+-\d+-\d+$', str(cas or '')))

    if cas_valid:
        compounds = pcp.get_compounds(cas, 'name')
        if compounds:
            return compounds[0].smiles

    # Fallback: search by substance name (stripped of qualifiers)
    compounds = pcp.get_compounds(name, 'name')
    if compounds:
        return compounds[0].smiles

    return None


results = []
total = len(df_all)

for _, row in df_all.iterrows():
    smiles = fetch_smiles(row['cas'], row['name'])
    results.append({'name': row['name'], 'xml_name': row['xml_name'],
                    'cas': row['cas'], 'smiles': smiles})

    n = len(results)
    if n % 50 == 0:
        found = sum(1 for r in results if r['smiles'])
        print(f'{n}/{total} processed — {found} SMILES found')

df_results = pd.DataFrame(results)
found = df_results['smiles'].notna().sum()
print(f'\nDone. {found}/{len(df_results)} substances have SMILES.')


/tmp/ipykernel_240325/968967552.py:13: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  return compounds[0].isomeric_smiles
/tmp/ipykernel_240325/968967552.py:8: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  return compounds[0].isomeric_smiles


50/679 processed — 39 SMILES found


## Step 5 — Inspect and Save

In [ ]:
df_results.head(10)

In [ ]:
# Substances without SMILES — typically mixtures, metabolite sums, or not in PubChem
df_missing = df_results[df_results['smiles'].isna()]
print(f'Missing SMILES ({len(df_missing)}):')
df_missing[['name', 'cas']]

In [ ]:
output_path = 'pesticides_smiles.csv'
df_results.to_csv(output_path, index=False)
print(f'Saved {len(df_results)} rows to {output_path}')